# COS720 – AI-Powered Insider Threat Detection System
## Notebook 02: Preprocessing & Feature Engineering

**Objective:** Clean the dataset, encode categorical variables, handle class imbalance using SMOTE, and prepare train/test splits ready for model training.

---
### Steps
1. Load and drop the constant feature (`late_exit_flag`)
2. Encode categorical variables with Label Encoding
3. Split data (80/20, stratified)
4. Apply SMOTE to the training set to handle class imbalance
5. Scale numeric features
6. Save all artifacts for downstream notebooks

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, os, warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

DATASET_PATH = 'insider_threat_clean_dataset.csv'
MODELS_DIR   = 'models/'
RANDOM_STATE = 42

os.makedirs(MODELS_DIR, exist_ok=True)
print('Setup complete.')

## 2. Load Dataset

In [ ]:
df = pd.read_csv(DATASET_PATH, sep=';')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

## 3. Drop Constant Feature

`late_exit_flag` was identified in data exploration as a zero-variance feature (always 0). It contributes no predictive signal and must be removed.

In [ ]:
print(f"late_exit_flag unique values: {df['late_exit_flag'].unique()}")
df = df.drop(columns=['late_exit_flag'])
print(f'After dropping constant feature: {df.shape[1]} columns remain')

## 4. Encode Categorical Features

We use **Label Encoding** because:
- Tree-based models (Random Forest, Decision Tree) handle ordinal-encoded categoricals well
- One-Hot Encoding would create 90+ columns for `employee_position` (48 unique values), significantly increasing dimensionality
- The encoders are saved so the prototype can encode new inputs identically

In [ ]:
cat_cols = ['employee_department', 'employee_campus',
            'employee_position', 'employee_origin_country']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f'{col}: {len(le.classes_)} classes → encoded as integers 0–{len(le.classes_)-1}')

# Persist encoders
with open(f'{MODELS_DIR}encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

print('\nEncoders saved.')
df.head(3)

## 5. Feature / Target Split

In [ ]:
X = df.drop(columns=['is_malicious'])
y = df['is_malicious']

feature_names = list(X.columns)

print(f'Features ({len(feature_names)}): {feature_names}')
print(f'Target:  is_malicious  →  {dict(y.value_counts())}')

# Save feature names for prototype
with open(f'{MODELS_DIR}feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

## 6. Train / Test Split (Stratified 80/20)

Stratification ensures the ~5.4% malicious class proportion is preserved in both splits.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f'Training set:  {len(X_train):,} rows | Malicious: {y_train.sum():,} ({y_train.mean()*100:.2f}%)')
print(f'Test set:      {len(X_test):,} rows  | Malicious: {y_test.sum():,}  ({y_test.mean()*100:.2f}%)')

## 7. Handling Class Imbalance with SMOTE

**SMOTE (Synthetic Minority Over-sampling Technique)** generates synthetic samples for the minority class (malicious) by interpolating between existing minority samples. 

**Why SMOTE?**
- Simple random oversampling risks overfitting by duplicating identical records
- Undersampling the majority class (94.6%) would discard ~106,000 records
- SMOTE creates plausible new minority samples, giving the model more signal
- Applied **only to the training set** — the test set remains unmodified to reflect real-world distribution

In [ ]:
print('Before SMOTE:')
print(f'  Normal (0):    {(y_train==0).sum():,}')
print(f'  Malicious (1): {(y_train==1).sum():,}')

sm = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print('\nAfter SMOTE:')
print(f'  Normal (0):    {(y_train_res==0).sum():,}')
print(f'  Malicious (1): {(y_train_res==1).sum():,}')
print(f'  Total:         {len(X_train_res):,}')

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, counts, title in [
    (axes[0], pd.Series(y_train).value_counts(),     'Before SMOTE'),
    (axes[1], pd.Series(y_train_res).value_counts(), 'After SMOTE')
]:
    ax.bar(['Normal', 'Malicious'], counts.values, color=['#4C72B0','#DD8452'], edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 200, f'{v:,}', ha='center', fontsize=9)
plt.suptitle('Class Balance: Training Set', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Feature Scaling

StandardScaler is fitted on the resampled training data and applied to both train and test sets. This is required for Logistic Regression (distance-based) but also applied uniformly for consistency across model comparisons.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_res)
X_test_sc  = scaler.transform(X_test)

print('Scaling complete.')
print(f'Training scaled shape: {X_train_sc.shape}')
print(f'Test scaled shape:     {X_test_sc.shape}')

with open(f'{MODELS_DIR}scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Scaler saved.')

## 9. Save Processed Data

In [ ]:
# Save splits for model notebook
splits = {
    'X_train': X_train_res,
    'X_train_sc': X_train_sc,
    'y_train': y_train_res,
    'X_test': X_test,
    'X_test_sc': X_test_sc,
    'y_test': y_test,
}
with open(f'{MODELS_DIR}splits.pkl', 'wb') as f:
    pickle.dump(splits, f)

# Also save test set as CSV for testing notebook
X_test_out = pd.DataFrame(X_test, columns=feature_names)
X_test_out['is_malicious'] = y_test.values
X_test_out.to_csv(f'{MODELS_DIR}X_test.csv', index=False)

print('All preprocessing artifacts saved to models/ directory:')
for f in os.listdir(MODELS_DIR):
    print(f'  {f}')

## 10. Preprocessing Summary

| Step | Action | Justification |
|---|---|---|
| Drop `late_exit_flag` | Removed | Zero-variance constant — no predictive value |
| Label Encoding | 4 categorical columns encoded | Tree models handle integers natively; avoids 90+ OHE columns |
| Stratified split | 80/20 train/test | Preserves the 5.4% malicious class ratio in both sets |
| SMOTE | Training set balanced 1:1 | Prevents model bias toward majority class |
| StandardScaler | All numeric features scaled | Required for Logistic Regression; applied uniformly for fair comparison |

**Proceed to Notebook 03 – Model Development.**